This notebook allows to search OpenAlex using a specific search phrase, and embed and store the fecthed articles into a chroma database.

In [1]:
import platform
import requests

# Fetch articles from OpenAlex

##### Functions for OpenAlex search

In [2]:
import math

#function to count pages and number of results (without fetching)
def get_total_pages(query, per_page=200):
    base_url = "https://api.openalex.org/works"  # Replace with the actual API endpoint
    # Create filters
    filters = [
        "has_abstract:true",
        "has_fulltext:true"
        #f"from_publication_date:{min_year}-01-01"
    ]
    
    params = {
        "search": query,
        "filter": ",".join(filters) if filters else None,
        "per_page": 1,  # Fetch only one result to get metadata
    }
    
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    data = response.json()
    
    total_results = data.get("meta", {}).get("count", 0)
    total_pages = math.ceil(total_results / per_page)  # Calculate total pages
    
    return total_results, total_pages


# function to parse the text
def reconstruct_text(inverted_index):
    word_index = []
    for k,v in inverted_index.items():
        for index in v:
            word_index.append([k,index])

    word_index = sorted(word_index,key = lambda x : x[1])

    word_list = []
    for i in range(len(word_index)):
        word_list.append(word_index[i][0])

    separator = ' '
    reconstructed_text = separator.join(word_list)

    return reconstructed_text

# function that uses openalex to search web for papers
def search_openalex(search_phrase):
    base_url = "https://api.openalex.org/works"  # Replace with the actual API endpoint

    # Create filters
    filters = [
        "has_abstract:true",
        "has_fulltext:true"
        #f"from_publication_date:{min_year}-01-01"
    ]

    # Construct the query parameters
    params = {
    "search": search_phrase,
    "filter": str.join(",", filters),  # Only return works with abstracts
    "per_page": 200,  # maximum allowed per page ,
    "cursor": "*", 
    }

    #fetch articles from all pages
    all_results = []
    abstract_list = []

    i = 0
    while params["cursor"]:
        print("page " + str(i))
        r = requests.get(base_url, params=params)
        r.raise_for_status()
        res_json = r.json()
        
        # Add the current batch of results
        all_results.extend(res_json.get("results", []))
        
        # Update the cursor to fetch the next page
        params["cursor"] = res_json.get("meta", {}).get("next_cursor")
        i = i+1
        
        for j in range(len(res_json["results"])):
            abstract_list.append(reconstruct_text(res_json["results"][j]['abstract_inverted_index']))

    return all_results, abstract_list

##### OpenAlex Search

In [3]:
search_phrase = '("mCRPC" OR "metastatic CRPC" OR "metastatic castration-resistant prostate cancer" \
OR "metastatic castration-resistant prostate carcinoma")'

In [4]:
#collect 
total_results, total_pages = get_total_pages(search_phrase)
print(f"Total results: {total_results}")
print(f"Total pages: {total_pages}")

res_, abstract = search_openalex(search_phrase)
print(f"Total results fetched: {len(res_)}")

Total results: 10829
Total pages: 55
page 0
page 1
page 2
page 3
page 4
page 5
page 6
page 7
page 8
page 9
page 10
page 11
page 12
page 13
page 14
page 15
page 16
page 17
page 18
page 19
page 20
page 21
page 22
page 23
page 24
page 25
page 26
page 27
page 28
page 29
page 30
page 31
page 32
page 33
page 34
page 35
page 36
page 37
page 38
page 39
page 40
page 41
page 42
page 43
page 44
page 45
page 46
page 47
page 48
page 49
page 50
page 51
page 52
page 53
page 54
page 55
Total results fetched: 10830


# Embed end store articles in chromaDB

In [5]:
!pip install chromadb
!pip install sentence_transformers

'pip' n'est pas reconnu en tant que commande interne
ou externe, un programme ex�cutable ou un fichier de commandes.
'pip' n'est pas reconnu en tant que commande interne
ou externe, un programme ex�cutable ou un fichier de commandes.


In [ ]:
#create or get collection

import chromadb
from chromadb.utils import embedding_functions

CHROMA_DATA_PATH = "chroma_data_20250603/"
EMBED_MODEL =  "all-MiniLM-L6-v2"
COLLECTION_NAME = "searchable_db_collection_fd"

client = chromadb.PersistentClient(path=CHROMA_DATA_PATH)
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)

existing_collections = client.list_collections()
print(existing_collections)
if COLLECTION_NAME not in [str(col) for col in existing_collections]:
    collection = client.create_collection(
                                        name=COLLECTION_NAME,
                                        embedding_function=embedding_func,
                                        metadata={"hnsw:space": "cosine"})
else:
    collection = client.get_collection(name="searchable_db_collection_fd")

c:\Users\thiba\anaconda3\envs\evidence_db2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[]


In [7]:
#get titles and metadata, create IDs
string_ids = [str(i) for i in range(len(res_))]  # Generating unique string IDs
titles = [item["title"] or "Unknown Title" for item in res_]
metadatas = []
#Print All available metadata 
for item in res_:
    # print(item["id"])
    # print(item.keys())
    # print(item["title"])
    # print(item["authorships"])
    # print(item["publication_year"])
    # print(item["primary_location"])
    # print(item["primary_location"]["source"])
    # print(item["primary_location"]["source"]["display_name"])
    country=""
    if item["authorships"]:
        if len(item["authorships"])>0:
            if len(item["authorships"][0]["countries"]) > 0:
               country=item["authorships"][0]["countries"][0]
               print(country)
    metadatas.append({
        "titles": item["title"] or "Unknown Title",
        "first_author": (
            item.get("authorships", [{}])[0].get("author", {}).get("display_name")
            if item.get("authorships") and len(item.get("authorships")) > 0
            else "Unknown Author"  # Fallback to "Unknown Author" if authorships is None or empty
        ),  # Fallback to "Unknown Author" if display_name is None
        "journal": (
            (item.get("primary_location", {}).get("source", {}) or {}).get("display_name")
            or "Unknown Journal"
        ),   # Fallback to "Unknown Journal" if display_name is None
        "year": item["publication_year"] or "Unknown Publication Year",
        "openAlex_id": item["id"] or "Unknown OpenAlex ID",
        "countryMainAuthor": country
    #  "bestOAUrl": item["best_oa_location"]["landing_page_url"]   
    })




#if test==0:
#    print(metadatas)
#    test=test+1

GB
US
GB
US
US
US
US
US
US
US
FR
US
GB
GB
DE
US
DE
DE
US
US
FR
DK
IT
US
US
IN
US
DE
DE
US
US
US
DE
CA
US
US
IE
US
DE
US
US
SE
DE
ES
US
US
PL
DK
CA
FR
DE
CA
FR
US
FR
US
AU
ES
US
US
US
FR
DE
US
IN
NL
US
US
BE
GB
NL
US
US
ZA
CA
NO
DK
GB
DE
KR
DE
GB
AU
AR
NL
AT
FR
US
US
GB
US
IL
US
DE
US
US
US
US
GB
BE
US
CA
GB
AU
CA
US
DE
US
CA
US
GB
GB
US
CA
AU
GB
US
US
US
IT
US
US
DK
US
US
GB
US
CA
US
US
US
US
CZ
US
US
DK
US
US
CN
US
US
US
GB
US
FR
IN
GB
US
GB
US
US
NL
DE
FR
IT
FR
US
AU
CN
SE
US
US
CN
US
US
US
FR
US
US
NL
US
US
GB
NL
FR
IT
US
FR
GB
HK
CA
US
DE
US
US
DE
US
US
NL
NL
US
GB
US
US
AU
CN
US
US
US
GB
CA
US
US
US
CA
US
CA
US
US
US
US
ES
US
US
GB
FR
US
SE
FR
GB
CA
US
US
LB
US
GB
FR
GB
US
US
US
ES
DE
DE
US
US
US
US
GB
US
SE
US
US
NO
US
IT
IT
DE
US
US
US
CN
US
IT
CN
US
FR
IN
US
JP
CH
SE
DE
NL
CA
US
LB
ES
GB
US
CN
US
TW
DE
GB
IT
US
US
US
CA
ES
US
AU
US
DE
DE
US
IT
US
NL
US
AU
NL
US
US
US
CN
CN
US
FR
CA
GB
DE
US
US
US
FR
US
US
US
US
JP
IT
CN
HK
IT
US
US
IT
US
US
AU
US
US
US
US
US
DE
US
FR
US
US
JP
U

In [8]:
print(metadatas[5])

{'titles': 'Integrating evolutionary dynamics into treatment of metastatic castrate-resistant prostate cancer', 'first_author': 'Jingsong Zhang', 'journal': 'Nature Communications', 'year': 2017, 'openAlex_id': 'https://openalex.org/W2770572337', 'countryMainAuthor': 'US'}


In [9]:
from tqdm import tqdm  # Import the progress bar library

# Define batch size (tuning this can improve performance)
batch_size = 10  # Adjust batch size as needed

# Get the total number of documents
total_docs = len(res_)

# Loop through the data in chunks and add to the collection
for i in tqdm(range(0, total_docs, batch_size), desc="Adding data to collection", unit="batch"):
    # Create a chunk of data
    start_idx = i
    end_idx = min(i + batch_size, total_docs)
    
    documents_chunk = [titles[j] + " " + abstract[j] for j in range(start_idx, end_idx)]
    ids_chunk = string_ids[start_idx:end_idx]
    metadatas_chunk = metadatas[start_idx:end_idx]
    
    # Add the current chunk to the collection
    collection.add(
        documents=documents_chunk,
        ids=ids_chunk,
        metadatas=metadatas_chunk
    )


Adding data to collection: 100%|██████████| 1083/1083 [04:47<00:00,  3.77batch/s]


In [11]:
# In make_chromadb.ipynb, after the tqdm loop for collection.add:
print("\n--- Verification Step ---")
if collection.count() > 0:
    if not string_ids: # Make sure string_ids is populated from your data loading
        print("  ERROR: string_ids list is empty. Cannot perform verification.")
    else:
        test_id_to_check = string_ids[0] 
        try:
            retrieved_item_notebook = collection.get(ids=[test_id_to_check], include=['embeddings', 'documents'])
            print(f"Retrieved item '{test_id_to_check}' from notebook context:")
            
            # Check documents
            doc_text_list = retrieved_item_notebook.get('documents')
            if doc_text_list and len(doc_text_list) > 0:
                doc_text = doc_text_list[0]
                print(f"  Document (first 100 chars): {doc_text[:100]}...")
            else:
                print(f"  Document for '{test_id_to_check}' is MISSING or EMPTY.")

            # Check embeddings
            embeddings_result = retrieved_item_notebook.get('embeddings') # This could be a list of embeddings or None
            
            actual_embedding_vector = None
            
            # Determine if embeddings_result is a list and extract the first embedding
            if isinstance(embeddings_result, list):
                if len(embeddings_result) > 0:
                    actual_embedding_vector = embeddings_result[0]
                else:
                    print(f"  ERROR: Embedding list for '{test_id_to_check}' is EMPTY (list of len 0).")
            elif hasattr(embeddings_result, 'shape'): # Check if it's a NumPy array directly (less common for .get() but possible)
                # This case might indicate an unexpected return structure or a single embedding returned not in a list
                print(f"  INFO: Embeddings result appears to be a direct NumPy array. Shape: {embeddings_result.shape}")
                if embeddings_result.ndim == 1: # A single flat vector
                     actual_embedding_vector = embeddings_result
                elif embeddings_result.ndim > 1 and embeddings_result.shape[0] == 1: # An array of arrays, take the first row
                     actual_embedding_vector = embeddings_result[0]
                else:
                    print(f"  ERROR: Embeddings result is a NumPy array with unexpected shape: {embeddings_result.shape}")
            
            if actual_embedding_vector is not None:
                embedding_length = 0
                # Check if it's a numpy array by checking for 'size' attribute (that isn't a method)
                is_numpy_array = hasattr(actual_embedding_vector, 'size') and not callable(getattr(actual_embedding_vector, 'size', None))
                is_list = isinstance(actual_embedding_vector, list)

                if is_numpy_array:
                    embedding_length = actual_embedding_vector.size 
                elif is_list: # Should be a flat list of numbers if it's an embedding
                    embedding_length = len(actual_embedding_vector)
                
                if embedding_length > 0:
                    print(f"  Embedding for '{test_id_to_check}' successfully generated. Type: {type(actual_embedding_vector)}, Length/Size: {embedding_length}")
                    try:
                        sample = actual_embedding_vector[:5] # Works for lists and numpy arrays
                        print(f"  Embedding sample (first 5 dims): {sample}")
                    except Exception as e_sample:
                        print(f"  Embedding sample could not be retrieved/sliced (type: {type(actual_embedding_vector)}): {e_sample}")
                else: # length is 0
                    print(f"  ERROR: Embedding for '{test_id_to_check}' is EMPTY (length/size 0). Type: {type(actual_embedding_vector)}")
            elif isinstance(embeddings_result, list) and not embeddings_result: 
                pass # Already handled: print(f"  ERROR: Embedding list for '{test_id_to_check}' is EMPTY (list of len 0).")
            else: # actual_embedding_vector remained None and wasn't an empty list
                print(f"  ERROR: Embedding for '{test_id_to_check}' is MISSING or in an unexpected format. Embeddings result from DB: {embeddings_result}")
        
        except Exception as e:
            print(f"  ERROR during verification for item '{test_id_to_check}': {e}")
            import traceback
            traceback.print_exc()
else:
    print("Collection is empty after population attempt. No items to verify.")
print("--- End Verification Step ---\n")


--- Verification Step ---
Retrieved item '0' from notebook context:
  Document (first 100 chars): Olaparib for Metastatic Castration-Resistant Prostate Cancer Multiple loss-of-function alterations i...
  INFO: Embeddings result appears to be a direct NumPy array. Shape: (1, 384)
  Embedding for '0' successfully generated. Type: <class 'numpy.ndarray'>, Length/Size: 384
  Embedding sample (first 5 dims): [-0.00396646 -0.03565611  0.04440904 -0.06399134 -0.020214  ]
--- End Verification Step ---

